# 02 — Final corpus integrity, verified tool episodes, and modality gate

Notebooks 03 and 04 own their streaming crawlers. This gate checks real usable train/validation/test payload coverage and split disjointness. It does not require all configured terabyte-scale sources to be exhausted before the first joint run; remaining pinned cursors and pending rows stay in the corpus artifact for later extension. Synthetic tool episodes remain separate and never satisfy external-source quotas.


In [ ]:
import os, sys, json
from pathlib import Path

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        for project in (candidate, candidate / "onecore-ai"):
            if (project / "configs" / "phone-research.yaml").is_file() and (project / "src" / "onecore").is_dir():
                return project
    raise RuntimeError(f"Could not locate OneCore project root from {current}")

root = find_project_root()
print("OneCore project root:", root)
sys.path.insert(0, str(root / "src"))
from onecore.acquire import acquire_sources, assert_task_coverage, external_asset_report_by_split
from onecore.config import load_config
_, _, project_config = load_config(root / "configs" / "phone-research.yaml")
data_gate_config = project_config.get("data_gate", {})
from onecore.crawlers import notebook_cursor_summary
from onecore.data import validate_training_data, validate_validation_data, validate_test_data, validate_split_disjointness
from onecore.tool_episodes import generate_tool_episodes

report_path = root / "data" / "acquisition-report.json"
if os.environ.get("ONECORE_SKIP_ACQUISITION", "0") == "1":
    if not report_path.is_file():
        raise FileNotFoundError("restored corpus is missing acquisition-report.json")
    acquisition_report = json.loads(report_path.read_text(encoding="utf-8"))
else:
    budget_seconds = int(os.environ.get("ONECORE_ACQUIRE_BUDGET_SECONDS", "0"))
    acquisition_report = acquire_sources(
        root,
        profile=os.environ.get("ONECORE_DATA_PROFILE", "initial"),
        time_budget_seconds=budget_seconds or None,
    )

required_default = ",".join(data_gate_config.get("required_families", ["text", "tool", "image", "audio", "video"]))
required = {item.strip() for item in os.environ.get("ONECORE_REQUIRED_TASKS", required_default).split(",") if item.strip()}
complete = bool(acquisition_report.get("acquisition_complete", True))
finalize_gate = os.environ.get("ONECORE_FINALIZE_DATA_GATE", "1") == "1"
if finalize_gate:
    # The acquisition report may be resumed and contain a stale pre-finalization
    # asset snapshot. Recompute it from the final split manifests every time.
    acquisition_report["external_media_assets"] = external_asset_report_by_split(root)
    if required:
        coverage = assert_task_coverage(
            root, required,
            minimum_train=int(os.environ.get("ONECORE_MIN_TRAIN_RECORDS", str(data_gate_config.get("min_train_per_family", 100)))),
            minimum_holdout=int(os.environ.get("ONECORE_MIN_HOLDOUT_RECORDS", str(data_gate_config.get("min_holdout_per_family", 10)))),
        )
        acquisition_report["task_coverage"] = coverage
    acquisition_report["training_ready"] = True
    acquisition_report["training_ready_task_groups"] = sorted(required)
    acquisition_report["training_ready_coverage"] = acquisition_report.get("task_coverage", {})
    # Add small, verified multi-step tool practice only after external task coverage
    # has been measured. These synthetic records never satisfy external source quotas.
    acquisition_report["verified_tool_episodes"] = generate_tool_episodes(root)
    report_path.write_text(json.dumps(acquisition_report, ensure_ascii=False, indent=2), encoding="utf-8")

print("Acquisition summary:", json.dumps({
    "source_count": acquisition_report.get("source_count"),
    "new_records": acquisition_report.get("new_records"),
    "total_new_records": acquisition_report.get("total_new_records"),
    "task_counts": acquisition_report.get("task_counts"),
    "task_coverage": acquisition_report.get("task_coverage"),
    "completed_sources": acquisition_report.get("completed_sources"),
    "unavailable_sources": acquisition_report.get("unavailable_sources"),
    "pending_sources": acquisition_report.get("pending_sources"),
    "acquisition_complete": complete,
    "training_ready": acquisition_report.get("training_ready", False),
    "session_budget_exhausted": acquisition_report.get("session_budget_exhausted"),
    "metadata_only_sources": acquisition_report.get("metadata_only_sources"),
    "sources_with_access_required": acquisition_report.get("sources_with_access_required"),
    "sources_with_missing_usable_payloads": acquisition_report.get("sources_with_missing_usable_payloads"),
    "crawler_cursors": {
        "03_text_pretraining": notebook_cursor_summary(root, "03_text_pretraining"),
        "04_multimodal_curriculum": notebook_cursor_summary(root, "04_multimodal_curriculum"),
    },
    "sources": [{"source": item.get("source"), "status": item.get("status"), "read": item.get("read"), "accepted": item.get("accepted"), "errors": item.get("errors")} for item in acquisition_report.get("sources", [])],
}, ensure_ascii=False, indent=2))

if finalize_gate:
    train_report = validate_training_data(root)
    validation_report = validate_validation_data(root)
    test_report = validate_test_data(root)
    disjointness_report = validate_split_disjointness(root)
    report = {"train": train_report, "validation": validation_report, "test": test_report, "split_disjointness": disjointness_report}
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print("Dataset gates: PASS — five-family real payload coverage and split integrity verified.")
    if not complete:
        print("Upstream source cursors remain resumable; this is a partial stream, not a full mirror of the terabyte-scale collections.")
elif complete:
    print("All enabled source quotas are complete. The dedicated final gate validated the full corpus.")
else:
    print("Acquisition remains incomplete; this session preserved notebook-specific stream cursors and data shards for the next bounded run.")

